# Notebook 04 — Semantic Analysis: PCA, Trajectories, and Diversity (Figures 2–4)

**Paper:** *Graph-Structured Reinforcement Learning for Scientific Hypothesis Generation in Materials Design*

This notebook implements **Sections 2.2 and 2.3** of the paper, producing Figures 2, 3, and 4.

**Embedding model:** `google/embeddinggemma-300m` (768-dimensional sentence embeddings)  
**Dimensionality reduction:** PCA (2D projection for visualization)  
**Density estimation:** Gaussian KDE (contour plots)

| Section | Analysis | Figure |
|---|---|---|
| 2.2.1 | PCA + KDE contour plots of reasoning phases | Fig 2 |
| 2.2.2 | Directed trajectory plots (phase-to-phase transitions) | Fig 3 |
| 2.3   | Inter-phase centroid distance (semantic diversity) | Fig 4 |

**Input data:** `data/results/*_data_eval_all.jsonl`  
**Output figures:** `figures/`  
**Prerequisites:** `pip install sentence-transformers scikit-learn scipy matplotlib seaborn`

## 1. Imports and Paths

In [ ]:
import os
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from itertools import combinations
from pathlib import Path

from scipy.stats import gaussian_kde
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA

# Paths
DATA_DIR    = Path("../data/results")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

# Reasoning phases in the order they appear in the model output
PHASES       = ["brainstorm", "graph", "patterns", "synthesis"]
PHASE_LABELS = ["<brainstorm>", "<graph>", "<patterns>", "<synthesis>"]

# Consistent colormap per reasoning phase (matches paper figures)
CMAPS = {
    "<brainstorm>": "Reds",
    "<graph>":      "Blues",
    "<patterns>":   "Greens",
    "<synthesis>":  "Purples",
    "qwen":         "Greys",
    "llama":        "Greys",
}


## 2. Load Embedding Model

The `google/embeddinggemma-300m` model maps text to 768-dimensional vectors (Section 2.2.1).

In [ ]:
from sentence_transformers import SentenceTransformer

# Load once and reuse for all models — first run will download ~300 MB
embedder = SentenceTransformer(
    "google/embeddinggemma-300m",
    trust_remote_code=True,
    device="cuda",   # use "cpu" if no GPU available
)
print(f"Embedding model loaded. Output dim: {embedder.get_sentence_embedding_dimension()}")


## 3. Helpers: Text Extraction and Embedding

For Graph-PRefLexOR models, reasoning traces are split by XML sentinel to extract per-phase text.
For baseline models (Qwen no-thinking, Llama), the full output is split into 4 equal sequential chunks.

In [ ]:
def extract_phases(text: str) -> dict:
    """Extract text within each Graph-PRefLexOR reasoning phase.

    Returns a dict mapping phase name -> content string (empty string if phase absent).
    """
    def get_block(s, tag):
        start_tag = f"<{tag}>"
        end_tag   = f"</{tag}>"
        i = s.find(start_tag)
        if i == -1:
            return ""
        j = s.find(end_tag, i)
        if j == -1:
            return ""
        return s[i + len(start_tag): j].strip()

    # Extract <think> content (the full reasoning trace)
    think = get_block(text, "think")
    return {phase: get_block(think, phase) for phase in PHASES}


def split_into_chunks(text: str, n: int = 4) -> list:
    """Split text into n approximately equal sequential chunks (for baseline models)."""
    from nltk.tokenize import sent_tokenize
    sentences = sent_tokenize(text)
    if len(sentences) < n:
        sentences = text.split(".") if "." in text else [text]
    chunk_size = max(1, len(sentences) // n)
    chunks = []
    for i in range(n):
        start = i * chunk_size
        end   = (i + 1) * chunk_size if i < n - 1 else len(sentences)
        chunks.append(" ".join(sentences[start:end]))
    # Pad if fewer than n chunks
    while len(chunks) < n:
        chunks.append(chunks[-1] if chunks else "")
    return chunks[:n]


def embed_graph_model(df: pd.DataFrame, reasoning_col: str = "raw_output"):
    """Embed all reasoning phases for a Graph-PRefLexOR model.

    Returns:
        res_emb    : (N_sentences, 768) array of sentence-level embeddings
        res_labels : list of phase labels aligned with res_emb rows
        res_ids    : list of response IDs aligned with res_emb rows
        ans_emb    : (N_chunks, 768) array of answer chunk embeddings
        ans_labels : list of chunk labels ('chunk_1'...'chunk_4')
        ans_ids    : list of response IDs aligned with ans_emb rows
    """
    from nltk.tokenize import sent_tokenize

    res_texts, res_labels, res_ids = [], [], []
    ans_texts, ans_labels, ans_ids = [], [], []

    for _, row in df.iterrows():
        rid    = str(row.get("id", row.name))
        raw    = str(row.get(reasoning_col) or "")
        answer = str(row.get("answer") or row.get("raw_output") or "")

        # Reasoning: extract each phase, then tokenize into sentences
        phases = extract_phases(raw)
        for phase, content in phases.items():
            for sent in sent_tokenize(content):
                sent = sent.strip()
                if len(sent) > 20:
                    res_texts.append(sent)
                    res_labels.append(f"<{phase}>")
                    res_ids.append(rid)

        # Answer: split into 4 sequential chunks
        if "</think>" in raw:
            answer_text = raw.split("</think>")[-1].strip()
        else:
            answer_text = answer
        for i, chunk in enumerate(split_into_chunks(answer_text)):
            if len(chunk.strip()) > 20:
                ans_texts.append(chunk.strip())
                ans_labels.append(f"chunk_{i+1}")
                ans_ids.append(rid)

    res_emb = embedder.encode(res_texts, batch_size=64, show_progress_bar=True)
    ans_emb = embedder.encode(ans_texts, batch_size=64, show_progress_bar=True)

    return (np.array(res_emb), res_labels, res_ids,
            np.array(ans_emb), ans_labels, ans_ids)


def embed_baseline_model(df: pd.DataFrame, reasoning_col: str = "thinking"):
    """Embed reasoning trace and answer for a baseline (non-GRPO) model.

    Both reasoning trace and answer are split into 4 sequential chunks.
    """
    from nltk.tokenize import sent_tokenize

    res_texts, res_ids = [], []
    ans_texts, ans_ids = [], []

    for _, row in df.iterrows():
        rid     = str(row.get("id", row.name))
        raw     = str(row.get(reasoning_col) or row.get("raw_output") or "")
        answer  = str(row.get("answer") or "")

        for chunk in split_into_chunks(raw):
            if len(chunk.strip()) > 20:
                res_texts.append(chunk.strip())
                res_ids.append(rid)
        for chunk in split_into_chunks(answer):
            if len(chunk.strip()) > 20:
                ans_texts.append(chunk.strip())
                ans_ids.append(rid)

    res_emb = embedder.encode(res_texts, batch_size=64, show_progress_bar=True)
    ans_emb = embedder.encode(ans_texts, batch_size=64, show_progress_bar=True)

    return (np.array(res_emb), ["baseline"] * len(res_emb), res_ids,
            np.array(ans_emb), ["baseline"] * len(ans_emb), ans_ids)


## 4. Compute and Cache Embeddings

Embedding takes 5–15 minutes per model on GPU. Results are cached to disk.

In [ ]:
def load_or_compute_embeddings(model_name: str, data_file: Path,
                                is_graph: bool, cache_dir: Path = DATA_DIR,
                                reasoning_col: str = "raw_output"):
    """Load embeddings from cache or compute from scratch.

    Args:
        model_name:    Short name used for cache file naming (e.g. 'graph_8b').
        data_file:     JSONL file with model inference outputs.
        is_graph:      True for Graph-PRefLexOR models; False for baselines.
        cache_dir:     Directory where .pkl cache files are saved.
        reasoning_col: Column containing the reasoning trace text.
    """
    cache_file = cache_dir / f"embeddings_{model_name}.pkl"
    if cache_file.exists():
        print(f"Loading cached embeddings: {cache_file.name}")
        with open(cache_file, "rb") as f:
            return pickle.load(f)

    print(f"Computing embeddings for {model_name}...")
    df = pd.read_json(data_file, lines=True)

    if is_graph:
        result = embed_graph_model(df, reasoning_col=reasoning_col)
    else:
        result = embed_baseline_model(df, reasoning_col=reasoning_col)

    with open(cache_file, "wb") as f:
        pickle.dump(result, f)
    print(f"Saved cache: {cache_file.name}")
    return result


# ── Load embeddings for all models ──────────────────────────────────────────
# (Graph-PRefLexOR models)
graph_8b  = load_or_compute_embeddings("graph_8b",   DATA_DIR / "graph_8b_data_eval_all.jsonl",   is_graph=True)
graph_3b  = load_or_compute_embeddings("graph_3b",   DATA_DIR / "graph_3b_data_eval_all.jsonl",   is_graph=True)
graph_1_7b= load_or_compute_embeddings("graph_1_7b", DATA_DIR / "graph_1_7b_data_eval_all.jsonl", is_graph=True)

# (Baseline models — split into 4 sequential chunks)
qwen_8b   = load_or_compute_embeddings("qwen_8b",    DATA_DIR / "qwen_8b_data_eval_all.jsonl",    is_graph=False, reasoning_col="thinking")
qwen_1_7b = load_or_compute_embeddings("qwen_1_7b",  DATA_DIR / "qwen_1_7b_data_eval_all.jsonl",  is_graph=False, reasoning_col="thinking")
llama_3b  = load_or_compute_embeddings("llama_3b",   DATA_DIR / "llama_3b_data_eval_all.jsonl",   is_graph=False, reasoning_col="raw_output")

# Unpack tuples: (res_emb, res_labels, res_ids, ans_emb, ans_labels, ans_ids)
graph_8b_res_emb,  graph_8b_res_labels,  graph_8b_res_ids,  graph_8b_ans_emb,  graph_8b_ans_labels,  graph_8b_ans_ids  = graph_8b
graph_1_7b_res_emb,graph_1_7b_res_labels,graph_1_7b_res_ids,graph_1_7b_ans_emb,graph_1_7b_ans_labels,graph_1_7b_ans_ids= graph_1_7b
qwen_8b_res_emb,   qwen_8b_res_labels,   qwen_8b_res_ids,   qwen_8b_ans_emb,   qwen_8b_ans_labels,   qwen_8b_ans_ids   = qwen_8b
qwen_1_7b_res_emb, qwen_1_7b_res_labels, qwen_1_7b_res_ids, qwen_1_7b_ans_emb, qwen_1_7b_ans_labels, qwen_1_7b_ans_ids = qwen_1_7b
llama_3b_res_emb,  llama_3b_res_labels,  llama_3b_res_ids,  llama_3b_ans_emb,  llama_3b_ans_labels,  llama_3b_ans_ids  = llama_3b


## 5. Figure 2 — PCA + KDE Density Contour Plots (Section 2.2.1)

PCA projects embeddings to 2D. KDE estimates the density of reasoning steps, making the comparison invariant to the number of steps per model (which differs; Section 2.2.1).

In [ ]:
def plot_pca_kde(graph_emb, graph_labels, base_emb, graph_model_name, base_model_name,
                 title, save_path=None, phases=PHASE_LABELS):
    """Plot KDE contour density for Graph-PRefLexOR phases vs. a baseline.

    A shared PCA is fitted on all embeddings combined to ensure a common coordinate system.
    Each phase gets a separate KDE contour plot; the baseline is shown in grey.
    """
    # Fit PCA on combined embeddings (graph + baseline)
    combined = np.vstack([graph_emb, base_emb])
    pca = PCA(n_components=2)
    pca.fit(combined)

    graph_2d = pca.transform(graph_emb)
    base_2d  = pca.transform(base_emb)

    # Build a shared evaluation grid for KDE
    x_min = combined[:, 0].min() - 0.1 if combined.size else -1
    x_max = combined[:, 0].max() + 0.1 if combined.size else 1
    y_min = combined[:, 1].min() - 0.1 if combined.size else -1
    y_max = combined[:, 1].max() + 0.1 if combined.size else 1

    xx, yy = np.mgrid[x_min:x_max:100j, y_min:y_max:100j]
    grid   = np.vstack([xx.ravel(), yy.ravel()])

    fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
    ax.set_facecolor("white")
    fig.patch.set_facecolor("white")

    graph_labels_arr = np.array(graph_labels)

    # Plot one KDE contour per phase
    for phase in phases:
        mask = graph_labels_arr == phase
        pts  = graph_2d[mask]
        if len(pts) < 10:
            continue
        try:
            kde    = gaussian_kde(pts.T, bw_method=0.3)
            zz     = kde(grid).reshape(xx.shape)
            cmap   = CMAPS.get(phase, "Blues")
            levels = np.linspace(zz.max() * 0.1, zz.max(), 5)
            ax.contourf(xx, yy, zz, levels=levels, cmap=cmap, alpha=0.35)
            ax.contour( xx, yy, zz, levels=levels, cmap=cmap, linewidths=0.7)
        except Exception:
            pass

    # Baseline KDE in grey
    if len(base_2d) >= 10:
        try:
            kde_b = gaussian_kde(base_2d.T, bw_method=0.3)
            zz_b  = kde_b(grid).reshape(xx.shape)
            levels_b = np.linspace(zz_b.max() * 0.1, zz_b.max(), 5)
            ax.contourf(xx, yy, zz_b, levels=levels_b, cmap="Greys", alpha=0.25)
            ax.contour( xx, yy, zz_b, levels=levels_b, cmap="Greys", linewidths=0.7)
        except Exception:
            pass

    # Legend
    handles = [mpatches.Patch(color=plt.get_cmap(CMAPS.get(p, "Blues"))(0.6), label=p)
               for p in phases]
    handles.append(mpatches.Patch(color="grey", alpha=0.5, label=base_model_name))
    ax.legend(handles=handles, fontsize=7, loc="upper right")
    ax.set_xlabel("Dim-1", fontsize=9)
    ax.set_ylabel("Dim-2", fontsize=9)
    ax.set_title(title, fontsize=10)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()


# Figure 2 (a) — 8B reasoning traces
plot_pca_kde(graph_8b_res_emb, graph_8b_res_labels, qwen_8b_res_emb,
             "Graph-PRefLexOR-8B", "Qwen3-8B",
             "Graph-PRefLexOR 8B vs Qwen3-8B Reasoning Traces: PCA",
             save_path=FIGURES_DIR / "pca_reasoning_8b.png")

# Figure 2 (b) — 1.7B reasoning traces
plot_pca_kde(graph_1_7b_res_emb, graph_1_7b_res_labels, qwen_1_7b_res_emb,
             "Graph-PRefLexOR-1.7B", "Qwen3-1.7B",
             "Graph-PRefLexOR 1.7B vs Qwen3-1.7B Reasoning Traces: PCA",
             save_path=FIGURES_DIR / "pca_reasoning_1_7b.png")


## 6. Figure 3 — Directed Trajectory Plots (Section 2.2.2)

Each response is represented as a directed path in PCA space. For Graph-PRefLexOR, waypoints are phase centroids (brainstorm→graph→patterns→synthesis). For baselines, the response is split into 4 equal sequential chunks.

In [ ]:
def get_graph_trajectory(res_emb, res_labels, res_ids, phases=PHASE_LABELS):
    """Compute one centroid per phase per response for Graph-PRefLexOR.

    Returns a list of (N_phases, D) arrays, one per response.
    """
    res_emb_arr    = np.array(res_emb)
    res_labels_arr = np.array(res_labels)
    ids            = sorted(set(res_ids))
    trajectories   = []

    for rid in ids:
        mask   = np.array(res_ids) == rid
        r_emb  = res_emb_arr[mask]
        r_labs = res_labels_arr[mask]
        points = []
        for phase in phases:
            pm = r_labs == phase
            if pm.sum() > 0:
                points.append(r_emb[pm].mean(axis=0))
        if len(points) == len(phases):
            trajectories.append(np.stack(points))
    return trajectories


def get_chunk_trajectory(ans_emb, ans_ids, n_chunks=4):
    """Compute one embedding per sequential chunk per response (for baselines)."""
    ids  = sorted(set(ans_ids))
    trajs = []
    for rid in ids:
        mask  = np.array(ans_ids) == rid
        embs  = np.array(ans_emb)[mask]
        sz    = max(1, len(embs) // n_chunks)
        pts   = [embs[i*sz:(i+1)*sz].mean(axis=0) for i in range(n_chunks)]
        if len(pts) == n_chunks:
            trajs.append(np.stack(pts))
    return trajs


def plot_directed_trajectories(trajs1, trajs2, labels1, labels2, title,
                               save_path=None, colors1="#CC4444", colors2="#4444CC"):
    """Plot directed (arrowed) reasoning trajectories for two models in shared PCA space."""
    all_pts = np.vstack([t for trajs in [trajs1, trajs2] for t in trajs])
    pca = PCA(n_components=2)
    pca.fit(all_pts)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), dpi=200,
                                    sharey=True, sharex=True)
    fig.patch.set_facecolor("white")

    label_colors = dict(zip(PHASE_LABELS, ["#CC4444", "#4444CC", "#33AA55", "#BB55AA"]))

    def draw_trajs(ax, trajs, labels, model_name):
        ax.set_facecolor("white")
        all_2d = pca.transform(np.vstack(trajs))  # for axis limits

        for traj in trajs:
            pts_2d = pca.transform(traj)
            for i in range(len(pts_2d) - 1):
                ax.annotate("", xy=pts_2d[i+1], xytext=pts_2d[i],
                            arrowprops=dict(arrowstyle="->", color="salmon",
                                            lw=0.6, alpha=0.5))

        # Scatter coloured by phase/chunk label
        for j, (label, color) in enumerate(label_colors.items()):
            pts = np.vstack([pca.transform(t)[j:j+1] for t in trajs
                             if j < len(t)])
            ax.scatter(pts[:, 0], pts[:, 1], s=20, color=color,
                       label=label, alpha=0.7, zorder=3)

        ax.set_title(model_name, fontsize=11)
        ax.set_xlabel("PC-1", fontsize=9)
        ax.set_ylabel("PC-2", fontsize=9)
        ax.legend(fontsize=7, loc="upper right")

    draw_trajs(ax1, trajs1, labels1, labels1)
    draw_trajs(ax2, trajs2, labels2, labels2)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()


# Build trajectories for 8B comparison
g8b_res_trajs  = get_graph_trajectory(graph_8b_res_emb,  graph_8b_res_labels,  graph_8b_res_ids)
q8b_res_chunks = get_chunk_trajectory(qwen_8b_res_emb,   qwen_8b_res_ids)

# Figure 3 (a) — Directed reasoning trajectories
plot_directed_trajectories(
    g8b_res_trajs, q8b_res_chunks,
    "Graph-PRefLexOR-8B", "Qwen3-8B",
    title="Directed Reasoning Trajectories (8B): Graph-PRefLexOR vs Qwen3",
    save_path=FIGURES_DIR / "trajectories_reasoning_8b.png",
)


## 7. Figure 4 — Semantic Diversity: Inter-Phase Centroid Distance (Section 2.3)

For each response, the centroid of each reasoning phase is computed in 768-D space. All pairwise cosine distances between phase centroids are averaged to give a single per-response diversity score. Violin plots compare Graph-PRefLexOR vs baselines.

In [ ]:
def inter_phase_distances_graph(res_emb, res_labels, res_ids, phases=PHASE_LABELS):
    """Compute mean inter-phase centroid cosine distance per response (Graph-PRefLexOR)."""
    from scipy.spatial.distance import cosine
    res_emb_arr    = np.array(res_emb)
    res_labels_arr = np.array(res_labels)
    scores = []
    for rid in sorted(set(res_ids)):
        mask = np.array(res_ids) == rid
        r_emb  = res_emb_arr[mask]
        r_labs = res_labels_arr[mask]
        centroids = []
        for phase in phases:
            pm = r_labs == phase
            if pm.sum() > 0:
                centroids.append(r_emb[pm].mean(axis=0))
        if len(centroids) >= 2:
            # Average all pairwise cosine distances between phase centroids
            dists = [cosine(c1, c2) for c1, c2 in combinations(centroids, 2)]
            scores.append(np.mean(dists))
    return np.array(scores)


def inter_chunk_distances_baseline(ans_emb, ans_ids, n_chunks=4):
    """Compute mean inter-chunk centroid cosine distance per response (baselines)."""
    from scipy.spatial.distance import cosine
    scores = []
    for rid in sorted(set(ans_ids)):
        mask  = np.array(ans_ids) == rid
        embs  = np.array(ans_emb)[mask]
        sz    = max(1, len(embs) // n_chunks)
        centroids = [embs[i*sz:(i+1)*sz].mean(axis=0) for i in range(n_chunks)]
        if len(centroids) >= 2:
            dists = [cosine(c1, c2) for c1, c2 in combinations(centroids, 2)]
            scores.append(np.mean(dists))
    return np.array(scores)


def plot_diversity_violin(data_dict: dict, title: str, save_path=None):
    """Violin + jitter plot comparing semantic diversity distributions.

    Args:
        data_dict: {model_label: np.array_of_distances}
    """
    import seaborn as sns

    rows = []
    for label, values in data_dict.items():
        for v in values:
            rows.append({"Model": label, "Inter-Phase Centroid Distance": v})
    df_plot = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(len(data_dict) * 1.8, 5), dpi=150)
    sns.violinplot(data=df_plot, x="Model", y="Inter-Phase Centroid Distance",
                   palette=["#CC4444" if "Graph" in m else "#888888"
                            for m in data_dict.keys()],
                   inner=None, linewidth=0.8, ax=ax)
    # Jitter overlay
    sns.stripplot(data=df_plot, x="Model", y="Inter-Phase Centroid Distance",
                  color="black", size=2, alpha=0.4, jitter=True, ax=ax)

    # Mean markers
    for i, (label, values) in enumerate(data_dict.items()):
        ax.scatter([i], [values.mean()], marker="D", color="black", s=40, zorder=5)
        ax.errorbar([i], [values.mean()], yerr=[values.std()],
                    fmt="none", color="black", capsize=3, linewidth=1.5)
        # Horizontal median line
        ax.hlines(np.median(values), i - 0.3, i + 0.3, color="black", linewidth=1)

    ax.set_title(title, fontsize=11)
    ax.set_xlabel("")
    ax.set_ylabel("Inter-Phase Centroid Distance", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    plt.xticks(fontsize=9, rotation=10)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()


# Compute diversity scores
g8b_div   = inter_phase_distances_graph(graph_8b_res_emb,  graph_8b_res_labels,  graph_8b_res_ids)
q8b_div   = inter_chunk_distances_baseline(qwen_8b_res_emb, qwen_8b_res_ids)
g1_7b_div = inter_phase_distances_graph(graph_1_7b_res_emb, graph_1_7b_res_labels, graph_1_7b_res_ids)
q1_7b_div = inter_chunk_distances_baseline(qwen_1_7b_res_emb, qwen_1_7b_res_ids)

# Figure 4 (a) — Reasoning trace semantic diversity
plot_diversity_violin(
    {"Graph-PRefLexOR-8B": g8b_div,  "Qwen3-8B": q8b_div,
     "Graph-PRefLexOR-1.7B": g1_7b_div, "Qwen3-1.7B": q1_7b_div},
    title="Semantic Diversity in Reasoning Trace",
    save_path=FIGURES_DIR / "diversity_reasoning.png",
)

# Print Table 1 values
print("\nTable 1 — Mean semantic diversity:")
for name, vals in [("Reasoning trace 1.7B — Graph", g1_7b_div),
                   ("Reasoning trace 1.7B — Qwen",  q1_7b_div),
                   ("Reasoning trace 8B — Graph",   g8b_div),
                   ("Reasoning trace 8B — Qwen",    q8b_div)]:
    print(f"  {name}: {vals.mean():.2f} ± {vals.std():.2f}")
